# Lab 3: Karpathy's LLM-Wiki Pattern

**학습 목표**
- Andrej Karpathy의 "LLM Knowledge Base" 패턴 이해
- RAG와 근본적으로 다른 접근법: 검색이 아닌 컴파일
- 위키의 투명성, 축적성, 크로스 레퍼런스 장점 체험

**핵심 통찰**  
> "RAG는 매 질문마다 지식을 재발견한다.  
> Wiki는 지식을 한 번 컴파일하고 계속 축적한다."  
> — Andrej Karpathy, 2026.04

**소요 시간**: 40-50분  
**참고**: [Karpathy's Gist](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f)

In [ ]:
import sys
sys.path.insert(0, '..')

from rag_lab import KarpathyWiki, load_papers
from rag_lab.evaluate import Evaluator

## 1. 아키텍처 이해

```
┌─────────────────────────────────────────────────┐
│  Layer 1: raw/  (원본 논문 — 수정 불가)          │
└──────────────────────┬──────────────────────────┘
                       │ LLM이 읽고 컴파일
┌──────────────────────▼──────────────────────────┐
│  Layer 2: wiki/  (LLM이 생성/관리하는 위키)      │
│  ├── index.md       ← 탐색의 진입점             │
│  ├── paper_summary.md   ← 논문별 요약           │
│  ├── concept_X.md       ← 크로스커팅 개념       │
│  └── log.md             ← 작업 기록             │
└──────────────────────┬──────────────────────────┘
                       │ 질문 시: index → 관련 페이지 탐색
┌──────────────────────▼──────────────────────────┐
│  Answer: 위키 페이지 기반 종합 답변              │
└─────────────────────────────────────────────────┘
```

**Standard RAG와의 차이:**
- RAG: 문서 → 임베딩 → 벡터DB (불투명)
- Wiki: 문서 → 마크다운 위키 (사람이 읽을 수 있음!)

## 2. 위키 컴파일

In [ ]:
papers = load_papers()
print(f"논문 {len(papers)}편 로드\n")

# 위키 컴파일 (첫 실행 시 2-3분, 이후 캐시 사용)
wiki = KarpathyWiki(papers)

## 3. 위키 탐색

**핵심**: 위키의 모든 내용은 마크다운 파일로, 사람이 직접 읽고 편집할 수 있습니다.

In [ ]:
# 위키 페이지 목록
pages = wiki.list_pages()
print("위키 페이지 목록:")
for p in pages:
    print(f"  📄 {p}")

In [ ]:
# index.md 읽기 — 위키의 목차
print(wiki.read_page('index'))

In [ ]:
# 논문 요약 페이지 읽기
summary_pages = [p for p in pages if 'summary' in p]
if summary_pages:
    content = wiki.read_page(summary_pages[0])
    print(content[:1500])

In [ ]:
# 개념 페이지 읽기 — 여러 논문을 가로지르는 주제
concept_pages = [p for p in pages if 'concept_' in p]
print(f"크로스커팅 개념 {len(concept_pages)}개:\n")
for cp in concept_pages:
    content = wiki.read_page(cp)
    print(f"{'─' * 50}")
    print(content[:500])
    print()

### 🤔 RAG와 비교해보세요

| 관점 | Standard RAG | Karpathy Wiki |
|------|-------------|---------------|
| 저장 형태 | 벡터 (사람이 못 읽음) | 마크다운 (읽을 수 있음) |
| 지식 상태 | 매번 재발견 | 한번 컴파일, 계속 축적 |
| 크로스 레퍼런스 | 없음 | [[wiki links]] |
| 투명성 | 블랙박스 | 완전 투명 |
| 새 문서 추가 | 빠름 (임베딩만) | 느림 (위키 업데이트) |

## 4. 질의 (Query)

In [ ]:
# 단일 논문 질문
result = wiki.query("What are the key findings about intergenerational mobility?")

print(f"탐색한 페이지: {result.pages_navigated}개")
print(f"참조 페이지: {result.sources}")
print(f"총 시간: {result.total_time:.2f}초")
print(f"\n답변:\n{result.answer}")

In [ ]:
# 크로스 논문 질문 — 위키가 강한 영역
result = wiki.query(
    "Across all papers, what datasets are used? "
    "Which rely on natural experiments vs structural models?"
)

print(f"탐색한 페이지: {result.pages_navigated}개")
print(f"\n답변:\n{result.answer}")

## 5. 위키 건강 검사 (Lint)

Karpathy 패턴의 핵심 연산 중 하나: 위키의 품질을 주기적으로 점검합니다.

In [ ]:
lint_result = wiki.lint()

print(f"총 페이지: {lint_result['total_pages']}")
print(f"총 위키 링크: {lint_result['total_links']}")
print(f"고아 페이지: {lint_result['orphan_pages']}")
print(f"깨진 링크: {len(lint_result['broken_links'])}개")
if lint_result['broken_links']:
    print("\n깨진 링크 목록:")
    for bl in lint_result['broken_links'][:10]:
        print(f"  {bl['from']} → [[{bl['to']}]] (페이지 없음)")

## 6. 증분 업데이트 (Ingest)

새 논문을 위키에 추가하면 기존 페이지와 연결됩니다.

In [ ]:
# 예시: 가상의 새 논문 추가 (실제 실습에서는 papers/에 새 파일 추가)
# wiki.ingest_new("new_paper", "논문 텍스트...")

# 위키 로그 확인
print(wiki.read_page('log'))

## 7. 실험 과제

### 과제 A: 3가지 방법 비교
```python
from rag_lab import StandardRAG, LightRAG, EmbeddingEngine

engine = EmbeddingEngine()
rag = StandardRAG(papers, engine)
lrag = LightRAG(papers, engine)
# wiki는 이미 구축됨

question = "어떤 질문이든"
evaluator = Evaluator()
scores = evaluator.compare(question, {
    "RAG": rag.query(question).answer,
    "LightRAG": lrag.query(question).answer,
    "Wiki": wiki.query(question).answer,
})
evaluator.print_comparison(scores)
```

### 과제 B: 위키 페이지 직접 편집
```python
# wiki/ 폴더의 .md 파일을 직접 열어 편집하고
# 편집 후 같은 질문에 답변이 어떻게 변하는지 관찰
# → 이것이 Wiki 패턴의 핵심: 인간이 편집장!
```

### 과제 C: 스케일 한계 실험
```python
# papers/ 에 10편, 20편, 50편을 넣고 각 방법의
# 구축 시간과 답변 품질 변화를 기록하세요.
# → Wiki는 어느 시점에서 한계에 도달할까?
```